# Plots para o Pitch em Vídeo

Este notebook gera as figuras que aparecem no `PITCH.md` e adiciona outras úteis para a edição do vídeo. A parte de aplicações (clustering, contrafactual, line-up) já está em `analysis.ipynb` e não é repetida aqui.

**Estrutura:**
1. Motivação (Bell 2016: 85% equipe / 15% piloto)
2. Treinamento — evolução da loss e AUROC
3. Resultados finais — AUROC e métricas de teste
4. Ortogonalidade geométrica — cosseno por par (D-E, D-P, E-P)
5. Distribuição de |cos| por instância (entangled vs disentangled)
6. Cross-correlação dimensão-a-dimensão (heatmap)
7. Dependência não linear — nHSIC
8. Trade-off: desemaranhamento vs desempenho (Pareto)
9. Visualização 2D dos espaços latentes (PCA)

Todos os plots são salvos em `output/plots/` para uso direto na edição.

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from sklearn.decomposition import PCA

from IPython.display import display

sys.path.append(os.path.abspath('src'))

from train import prepare_data_and_graph, hsic_rbf, normalized_hsic
from models.pipeline_fusion import F1OrthogonalPipeline

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['axes.titleweight'] = 'bold'

PLOTS_DIR = Path('output/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_RESULTS_PATH = Path('output/models/training_results.json')
MODELS_DIR = Path('output/models')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PALETTE = {
    'model_no_orthogonal': '#bdbdbd',
    'model_orthogonal':    '#e15759',
    'model_ablation_p01_c01': '#f28e2b',
    'model_ablation_p1_c01':  '#ff9da7',
    'model_ablation_p1_c1':   '#9c755f',
    'model_hsic_01':          '#76b7b2',
    'model_hsic_1':           '#59a14f',
    'model_combo_p1_h01':     '#4e79a7',
}
PRETTY_NAME = {
    'model_no_orthogonal': 'Baseline (sem ortog.)',
    'model_orthogonal':    'Ortogonal (cos)',
    'model_ablation_p01_c01': 'cos 0.1 + cdim 0.1',
    'model_ablation_p1_c01':  'cos 1.0 + cdim 0.1',
    'model_ablation_p1_c1':   'cos 1.0 + cdim 1.0',
    'model_hsic_01':          'HSIC 0.1',
    'model_hsic_1':           'HSIC 1.0',
    'model_combo_p1_h01':     'cos 1.0 + HSIC 0.1',
}

def save_fig(name):
    path = PLOTS_DIR / f'{name}.png'
    plt.savefig(path)
    print(f'[ok] {path}')

with open(TRAINING_RESULTS_PATH) as f:
    results = json.load(f)
results_map = {r['model_name']: r for r in results}
available_models = [r['model_name'] for r in results if (MODELS_DIR / f"{r['model_name']}.pth").exists()]
print('Modelos disponíveis:', available_models)
print('Device:', device)

## 1. Motivação — Bell (2016): 85% equipe / 15% piloto

Citado no pitch como ponto de partida: a média histórica mostra que a maior parte da variância de desempenho vem da equipe. A pesquisa pergunta se conseguimos caracterizar piloto e equipe de forma totalmente desemaranhada.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
shares = [85, 15]
labels = ['Equipe (carro)', 'Piloto']
colors = ['#4e79a7', '#e15759']
wedges, texts, autotexts = ax.pie(
    shares, labels=labels, colors=colors,
    autopct='%1.0f%%', startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 3},
    textprops={'fontsize': 18, 'fontweight': 'bold'}
)
for at in autotexts:
    at.set_color('white')
    at.set_fontsize(22)
ax.set_title('Variância de desempenho na F1\n(Bell et al., 2016 — modelagem multinível)', pad=18)
save_fig('01_bell_variance_pie')
plt.show()

## 2. Treinamento — evolução de loss e AUROC

Decomposição da loss total (BCE + cosseno + cross-corr + HSIC) para o modelo ortogonal, e comparação de AUROC validação entre o baseline e o modelo ortogonal.

### Formulação da loss

Combinamos a perda da tarefa com três regularizadores complementares de desemaranhamento e duas cabeças auxiliares (piloto e equipe). Seja $\mathcal{P} = \{(v_\text{piloto}, v_\text{equipe}),\ (v_\text{piloto}, v_\text{pista}),\ (v_\text{equipe}, v_\text{pista})\}$ o conjunto dos três pares de espaços latentes. A loss total é:

$$
\boxed{\;
\mathcal{L}_{\text{total}}
\;=\;
\underbrace{\mathcal{L}_{\text{BCE}}(\hat{y}, y)}_{\text{tarefa}}
\;+\;
\alpha\!\left[\mathcal{L}_{\text{BCE}}(\hat{y}_\text{P}, y) + \mathcal{L}_{\text{BCE}}(\hat{y}_\text{E}, y)\right]
\;+\;
\lambda_{\cos}\, \mathcal{R}_{\cos}
\;+\;
\lambda_{\text{cd}}\, \mathcal{R}_{\text{cd}}
\;+\;
\lambda_{\text{H}}\, \mathcal{R}_{\text{HSIC}}
\;}
$$

com cada regularizador sendo a média sobre os pares $(u, v) \in \mathcal{P}$:

$$
\mathcal{R}_{\cos} \;=\; \frac{1}{|\mathcal{P}|}\!\!\sum_{(u,v)\in\mathcal{P}}\!
\mathbb{E}_i\!\left[\,\left|\,\tfrac{u_i^\top v_i}{\lVert u_i \rVert\, \lVert v_i \rVert}\,\right|\,\right]
\qquad
\mathcal{R}_{\text{cd}} \;=\; \frac{1}{|\mathcal{P}|}\!\!\sum_{(u,v)\in\mathcal{P}}\!
\frac{1}{d^{2}}\sum_{j,k}\left|\,C_{jk}(u,v)\,\right|
\qquad
\mathcal{R}_{\text{HSIC}} \;=\; \frac{1}{|\mathcal{P}|}\!\!\sum_{(u,v)\in\mathcal{P}}\!
\widehat{\text{HSIC}}_{\text{RBF}}(u, v)
$$

onde $C(u,v) \in \mathbb{R}^{d\times d}$ é a matriz de correlação cruzada *padronizada* entre as dimensões de $u$ e $v$ (espírito do **Barlow Twins**) e $\widehat{\text{HSIC}}_{\text{RBF}}$ é o estimador *biased* do Hilbert–Schmidt Independence Criterion com kernels gaussianos e largura via **mediana**.

### Parâmetros

| Símbolo | Nome no código | Papel |
|---|---|---|
| $\mathcal{L}_{\text{BCE}}(\hat{y}, y)$ | `loss_bce_main` | Binary cross-entropy da tarefa principal *driver-top3*. |
| $\hat{y}_\text{P},\ \hat{y}_\text{E}$ | `aux_piloto`, `aux_equipe` | Cabeças auxiliares lineares que tentam prever o alvo usando **só** $v_\text{piloto}$ ou **só** $v_\text{equipe}$ — evitam que um ramo domine o gradiente. |
| $\alpha$ | `aux_weight` (= 0.5) | Peso conjunto das cabeças auxiliares. |
| $\lambda_{\cos}$ | `lambda_pairwise` / `lambda_orthogonal` | Força da penalidade de **cosseno por amostra** — ataca o alinhamento geométrico local. |
| $\lambda_{\text{cd}}$ | `lambda_crossdim` | Força da **cross-correlação dimensão-a-dimensão** — ataca correlações lineares no estilo Barlow Twins. |
| $\lambda_{\text{H}}$ | `lambda_hsic` | Força do **HSIC com kernel RBF** — ataca dependências estatísticas *não lineares*. |
| $|\mathcal{P}|$ | — | Número de pares ativos (1 se sem pista, 3 com pista). |
| $d$ | `latent_dim` (= 8) | Dimensionalidade de cada espaço latente. |

Os três regularizadores são **complementares** — capturam fenômenos distintos:

- $\mathcal{R}_{\cos}$ → geometria **local** (cada amostra ortogonal por par).
- $\mathcal{R}_{\text{cd}}$ → desacoplamento **linear** dimensão-a-dimensão.
- $\mathcal{R}_{\text{HSIC}}$ → dependência **não linear** de distribuição.

In [ ]:
orth_hist = results_map['model_orthogonal']['history']
epochs = np.arange(1, len(orth_hist['train_loss']) + 1)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(epochs, orth_hist['train_loss'], marker='o', lw=2.5, color='#222', label='Loss total')
ax.plot(epochs, orth_hist['train_bce'],  marker='s', lw=2,   color='#4e79a7', label='BCE (tarefa)')
ax.plot(epochs, orth_hist['train_orth'], marker='^', lw=2,   color='#e15759', label='Cosseno (pares)')
ax.plot(epochs, orth_hist['train_crossdim'], marker='d', lw=2, color='#f28e2b', label='Cross-corr (dim)')
ax.plot(epochs, orth_hist['train_hsic_loss'], marker='v', lw=2, color='#59a14f', label='HSIC (batch)')
ax.set_xlabel('Época')
ax.set_ylabel('Valor da componente')
ax.set_title('Decomposição da loss no treino — modelo ortogonal')
ax.legend(loc='upper right', fontsize=11)
save_fig('02_loss_decomposition_orthogonal')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharex=True)

for name in ['model_no_orthogonal', 'model_orthogonal']:
    h = results_map[name]['history']
    color = PALETTE[name]
    axes[0].plot(epochs, h['train_loss'], marker='o', lw=2, color=color, label=f'{PRETTY_NAME[name]} (train)')
    axes[0].plot(epochs, h['val_loss'],   marker='s', lw=2, linestyle='--', color=color, label=f'{PRETTY_NAME[name]} (val)')
    axes[1].plot(epochs, h['val_auroc'],  marker='o', lw=2.5, color=color, label=PRETTY_NAME[name])

axes[0].set_title('Loss por época')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=10)

axes[1].set_title('AUROC de validação')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('AUROC')
axes[1].legend(fontsize=11, loc='lower right')
axes[1].set_ylim(0.45, 0.95)

save_fig('03_loss_auroc_evolution_baseline_vs_orthogonal')
plt.show()

## 3. Resultados finais — AUROC e métricas de teste

Tabela compacta e barra comparativa do AUROC no teste. No pitch:
> O modelo com ortogonalidade por cosseno atinge AUROC de teste de **0.852**, contra **0.831** do baseline.

In [ ]:
rows = []
for name in available_models:
    r = results_map[name]
    cfg = r['configuration']
    t = r['test_metrics']
    rows.append({
        'model': PRETTY_NAME.get(name, name),
        'key': name,
        'lambda_cos':    cfg.get('lambda_pairwise', 0.0),
        'lambda_crossdim': cfg.get('lambda_crossdim', 0.0),
        'lambda_hsic':   cfg.get('lambda_hsic', 0.0),
        'AUROC':        t['auroc'],
        '|cos| médio':  t['orth'],
        'cross-corr':   t['crossdim'],
        'nHSIC':        t['nhsic'],
    })
metrics_df = pd.DataFrame(rows).sort_values('AUROC', ascending=False).reset_index(drop=True)
display(metrics_df.style.format({
    'AUROC': '{:.3f}', '|cos| médio': '{:.3f}',
    'cross-corr': '{:.3f}', 'nHSIC': '{:.3f}',
}).background_gradient(subset=['AUROC'], cmap='Greens'))
metrics_df.to_csv(PLOTS_DIR / 'metrics_summary.csv', index=False)
print(f'[ok] {PLOTS_DIR / "metrics_summary.csv"}')

In [ ]:
order = ['model_no_orthogonal', 'model_orthogonal', 'model_ablation_p01_c01',
         'model_ablation_p1_c01', 'model_ablation_p1_c1',
         'model_hsic_01', 'model_hsic_1', 'model_combo_p1_h01']
order = [m for m in order if m in available_models]

auroc_vals = [results_map[m]['test_metrics']['auroc'] for m in order]
colors = [PALETTE[m] for m in order]
names = [PRETTY_NAME[m] for m in order]

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(names, auroc_vals, color=colors, edgecolor='black', linewidth=1.2)
for bar, val in zip(bars, auroc_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.003, f'{val:.3f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

baseline_v = results_map['model_no_orthogonal']['test_metrics']['auroc']
ax.axhline(baseline_v, color='#bdbdbd', linestyle='--', lw=1.5, alpha=0.8, label=f'Baseline = {baseline_v:.3f}')

ax.set_ylim(0.78, max(auroc_vals) + 0.02)
ax.set_ylabel('AUROC (teste)')
ax.set_title('AUROC no teste — driver-top3 benchmark')
ax.tick_params(axis='x', rotation=25)
for lbl in ax.get_xticklabels():
    lbl.set_ha('right')
ax.legend(loc='lower right')
save_fig('04_auroc_test_bar')
plt.show()

## 4. Ortogonalidade geométrica — cosseno por par

Para mostrar que a estrutura latente muda radicalmente, computamos |cos| médio em **todos os pares**: piloto–equipe, piloto–pista, equipe–pista — sobre o conjunto de teste.

Este é o gráfico explicitamente mencionado no pitch:
> O gráfico de cosseno entre pares deixa isso visível. Espaços que antes estavam emaranhados agora estão geometricamente separados.

In [ ]:
_, _, test_loader, graph_data = prepare_data_and_graph()
graph_data = graph_data.to(device)

_model_cache = {}

def load_model(name):
    if name in _model_cache:
        return _model_cache[name]
    cfg = results_map[name]['configuration']
    num_nodes_dict = {nt: graph_data[nt].num_nodes for nt in graph_data.node_types}
    model = F1OrthogonalPipeline(
        num_nodes_dict=num_nodes_dict,
        latent_dim=int(cfg.get('latent_dim', 8)),
        use_track_encoder=bool(cfg.get('use_track_encoder', True)),
    ).to(device)
    model.load_state_dict(torch.load(MODELS_DIR / f'{name}.pth', map_location=device))
    model.eval()
    _model_cache[name] = model
    return model

def collect_latents(name):
    model = load_model(name)
    vs_d, vs_e, vs_p = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            xd, xt, cid, _ = [b.to(device) for b in batch]
            _, _, _, vd, ve, vp = model(
                x_driver=xd, x_track=xt,
                graph_x_dict=graph_data.x_dict,
                graph_edge_index_dict=graph_data.edge_index_dict,
                target_constructor_ids=cid,
            )
            vs_d.append(vd.detach().cpu())
            vs_e.append(ve.detach().cpu())
            if vp is not None:
                vs_p.append(vp.detach().cpu())
    Vd = torch.cat(vs_d, dim=0)
    Ve = torch.cat(vs_e, dim=0)
    Vp = torch.cat(vs_p, dim=0) if vs_p else None
    return Vd, Ve, Vp

def pair_abs_cos(a, b):
    an = F.normalize(a, p=2, dim=-1)
    bn = F.normalize(b, p=2, dim=-1)
    return torch.abs((an * bn).sum(dim=-1)).numpy()

def pair_nhsic(a, b):
    return float(normalized_hsic(a, b).item())

latents = {name: collect_latents(name) for name in available_models}
print('Latentes coletados para:', list(latents.keys()))

In [ ]:
compare_models = ['model_no_orthogonal', 'model_orthogonal', 'model_hsic_1']
compare_models = [m for m in compare_models if m in available_models]

pair_keys = [('driver', 'team'), ('driver', 'track'), ('team', 'track')]
pair_labels = ['Piloto × Equipe', 'Piloto × Pista', 'Equipe × Pista']

data = {}
for m in compare_models:
    Vd, Ve, Vp = latents[m]
    vecs = {'driver': Vd, 'team': Ve, 'track': Vp}
    data[m] = [float(np.mean(pair_abs_cos(vecs[a], vecs[b]))) for a, b in pair_keys]

x = np.arange(len(pair_labels))
width = 0.26

fig, ax = plt.subplots(figsize=(12, 6))
for i, m in enumerate(compare_models):
    offset = (i - (len(compare_models) - 1) / 2) * width
    bars = ax.bar(x + offset, data[m], width=width,
                  color=PALETTE[m], edgecolor='black', linewidth=1.0,
                  label=PRETTY_NAME[m])
    for bar, val in zip(bars, data[m]):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.008, f'{val:.2f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(pair_labels)
ax.set_ylabel('|cos| médio por par')
ax.set_ylim(0, max(max(v) for v in data.values()) * 1.25)
ax.set_title('Cosseno médio entre pares de espaços latentes (teste)')
ax.legend(loc='upper right', fontsize=11)
save_fig('05_cosine_pairs_bar')
plt.show()

print('\nValores numéricos:')
display(pd.DataFrame(data, index=pair_labels).T.round(3))

## 5. Distribuição de |cos| por instância — entangled vs disentangled

Cada instância (corrida × piloto) produz um cosseno entre os vetores latentes. A distribuição mostra que com regularização o pico concentra-se perto de zero (ortogonalidade).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)

for ax, (a, b), title in zip(axes, pair_keys, pair_labels):
    for m in ['model_no_orthogonal', 'model_orthogonal']:
        if m not in latents:
            continue
        Vd, Ve, Vp = latents[m]
        vecs = {'driver': Vd, 'team': Ve, 'track': Vp}
        cos_vals = pair_abs_cos(vecs[a], vecs[b])
        ax.hist(cos_vals, bins=30, alpha=0.55, density=True,
                color=PALETTE[m], edgecolor='black', linewidth=0.6,
                label=PRETTY_NAME[m])
    ax.set_title(title)
    ax.set_xlabel('|cos|')
    ax.set_xlim(0, 1)
axes[0].set_ylabel('Densidade')
axes[0].legend(loc='upper right', fontsize=11)
fig.suptitle('Distribuição de |cos| por instância — baseline vs ortogonal', y=1.02, fontsize=18, fontweight='bold')
save_fig('06_cosine_histograms_per_pair')
plt.show()

## 6. Cross-correlação dimensão-a-dimensão

Heatmap de |corr| entre cada dimensão latente de piloto e cada dimensão de equipe. Atacar essas correlações é o trabalho do termo cross-corr (Barlow Twins).

In [ ]:
def cross_corr_matrix(a, b):
    eps = 1e-8
    a_c = a - a.mean(dim=0, keepdim=True)
    b_c = b - b.mean(dim=0, keepdim=True)
    a_s = a_c / (a_c.std(dim=0, keepdim=True) + eps)
    b_s = b_c / (b_c.std(dim=0, keepdim=True) + eps)
    n = a.size(0)
    return torch.abs((a_s.T @ b_s) / max(n - 1, 1)).cpu().numpy()

compare_cd = [m for m in ['model_no_orthogonal', 'model_orthogonal', 'model_ablation_p1_c1'] if m in available_models]
fig, axes = plt.subplots(1, len(compare_cd), figsize=(5.5 * len(compare_cd), 5.2))
if len(compare_cd) == 1:
    axes = [axes]
for ax, name in zip(axes, compare_cd):
    Vd, Ve, _ = latents[name]
    M = cross_corr_matrix(Vd, Ve)
    sns.heatmap(M, ax=ax, cmap='Reds', vmin=0, vmax=1, cbar=True,
                annot=False, square=True, linewidths=0.4, linecolor='white')
    ax.set_title(f'{PRETTY_NAME[name]}\nmédia |corr| = {M.mean():.3f}')
    ax.set_xlabel('dim V_equipe')
    ax.set_ylabel('dim V_piloto')
plt.tight_layout()
save_fig('07_crossdim_heatmaps')
plt.show()

## 7. Dependência não linear — nHSIC

Cosseno ataca alinhamento linear. nHSIC mede dependência estatística não linear. No pitch:
> Quando adicionamos a cross-correlação, o nHSIC global também cai, de **0.62** para **0.42**.

In [ ]:
nhsic_vals = [results_map[m]['test_metrics']['nhsic'] for m in order]
colors_n = [PALETTE[m] for m in order]
names_n = [PRETTY_NAME[m] for m in order]

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(names_n, nhsic_vals, color=colors_n, edgecolor='black', linewidth=1.2)
for bar, val in zip(bars, nhsic_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f'{val:.2f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')
baseline_n = results_map['model_no_orthogonal']['test_metrics']['nhsic']
ax.axhline(baseline_n, color='#bdbdbd', linestyle='--', lw=1.5, alpha=0.8, label=f'Baseline = {baseline_n:.2f}')
ax.set_ylabel('nHSIC (teste) — menor é melhor')
ax.set_title('Dependência não linear entre piloto e equipe')
ax.tick_params(axis='x', rotation=25)
for lbl in ax.get_xticklabels():
    lbl.set_ha('right')
ax.legend(loc='upper right')
ax.set_ylim(0, max(nhsic_vals) * 1.15)
save_fig('08_nhsic_test_bar')
plt.show()

## 8. Pareto — desemaranhamento vs desempenho

Dois scatters mostrando que ortogonalidade não custa AUROC. Pontos no canto **superior-esquerdo** são os melhores (alto AUROC, baixa dependência).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, metric, title in zip(
    axes,
    ['orth', 'nhsic'],
    ['|cos| médio (menor = mais ortogonal)', 'nHSIC (menor = menos dependência não linear)'],
):
    for name in order:
        t = results_map[name]['test_metrics']
        x = t[metric]; y = t['auroc']
        ax.scatter(x, y, s=260, color=PALETTE[name], edgecolor='black', linewidth=1.4, zorder=3)
        dx = (max([results_map[m]['test_metrics'][metric] for m in order]) -
              min([results_map[m]['test_metrics'][metric] for m in order])) * 0.012
        ax.annotate(PRETTY_NAME[name], (x, y), xytext=(x + dx, y + 0.001),
                    fontsize=10)
    ax.set_xlabel(title)
    ax.set_ylabel('AUROC (teste)')
    ax.grid(alpha=0.35)

axes[0].set_title('AUROC × ortogonalidade (cosseno)')
axes[1].set_title('AUROC × dependência não linear (nHSIC)')
save_fig('09_pareto_auroc_vs_disentanglement')
plt.show()

## 9. Visualização 2D dos espaços latentes (PCA)

Projeção dos vetores latentes em 2D pelo PCA. Visualmente: no baseline, piloto, equipe e pista se sobrepõem; com ortogonalidade, eles ocupam regiões distintas do plano.

In [ ]:
def project_latents_joint(Vd, Ve, Vp):
    arrs = [Vd.numpy(), Ve.numpy()]
    labels = ['Piloto'] * Vd.shape[0] + ['Equipe'] * Ve.shape[0]
    if Vp is not None:
        arrs.append(Vp.numpy())
        labels += ['Pista'] * Vp.shape[0]
    X = np.concatenate(arrs, axis=0)
    proj = PCA(n_components=2, random_state=0).fit_transform(X)
    return proj, np.array(labels)

pca_models = [m for m in ['model_no_orthogonal', 'model_orthogonal'] if m in available_models]
fig, axes = plt.subplots(1, len(pca_models), figsize=(7.2 * len(pca_models), 6), sharey=True)
if len(pca_models) == 1:
    axes = [axes]

block_colors = {'Piloto': '#e15759', 'Equipe': '#4e79a7', 'Pista': '#59a14f'}

for ax, name in zip(axes, pca_models):
    Vd, Ve, Vp = latents[name]
    proj, labels = project_latents_joint(Vd, Ve, Vp)
    for block, col in block_colors.items():
        mask = labels == block
        if not mask.any():
            continue
        ax.scatter(proj[mask, 0], proj[mask, 1], s=28, c=col, alpha=0.6,
                   edgecolor='white', linewidth=0.3, label=block)
    ax.set_title(PRETTY_NAME[name])
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=11)

fig.suptitle('Espaços latentes projetados em 2D (PCA conjunto)', y=1.02, fontsize=18, fontweight='bold')
save_fig('10_pca_latent_spaces')
plt.show()

## Resumo dos arquivos gerados

Todos os PNGs estão em `output/plots/`. Lista para uso direto na edição do vídeo:

In [ ]:
for p in sorted(PLOTS_DIR.glob('*')):
    print(f'- {p}')